# Notebook 3: Experiment 3 — Different Timeframe Training (80/20)## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction**Experiment:** Train on Weekly/Monthly/Yearly data, predict on Daily test data.  **Train/Test Split:** 80/20 (chronological, split date from daily data)  **Models:** BiLSTM, BiGRU, LSTM, GRU  **Scaler:** ProportionScaler (÷ 10,501)  **Metrics:** MSE, RMSE, MAE, MAPE, R² Score  

In [ ]:
import sys, osimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport warningswarnings.filterwarnings('ignore')sys.path.insert(0, '.')from stock_prediction_utils import *set_seed()set_ieee_style()DATA_DIR = '.'TRAIN_RATIO = 0.8RATIO_LABEL = '80_20'EXP_LABEL = f'Exp3_{RATIO_LABEL}'os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)os.makedirs('results', exist_ok=True)print(f"Experiment 3 - Different Timeframe Training (80/20)")

In [ ]:
# Load ALL timeframe dataprint("Loading daily data...")daily_data = load_all_daily_data(DATA_DIR)print("\nLoading weekly data...")weekly_data = {}for stock in STOCKS:    weekly_data[stock] = load_stock_data(get_data_paths(DATA_DIR, stock, 'weekly'))    print(f"  {stock}: {len(weekly_data[stock])} records")print("\nLoading monthly data...")monthly_data = {}for stock in STOCKS:    monthly_data[stock] = load_stock_data(get_data_paths(DATA_DIR, stock, 'monthly'))    print(f"  {stock}: {len(monthly_data[stock])} records")print("\nLoading yearly data...")yearly_data = {}for stock in STOCKS:    yearly_data[stock] = load_stock_data(get_data_paths(DATA_DIR, stock, 'yearly'))    print(f"  {stock}: {len(yearly_data[stock])} records")print("\nAll data loaded!")timeframe_data = {    'weekly': weekly_data,    'monthly': monthly_data,    'yearly': yearly_data,}

## Run All Timeframe Experiments

In [ ]:
# ============================================================# EXPERIMENT 3: Different timeframe training# ============================================================all_results = []for stock in STOCKS:    for tf_name in TIMEFRAMES:        print(f"\n{'#'*60}")        print(f"# STOCK: {stock} | TRAIN TIMEFRAME: {tf_name}")        print(f"{'#'*60}")                train_df = timeframe_data[tf_name][stock]        test_daily_df = daily_data[stock]                # Prepare data        X_train, y_train, X_test, y_test, test_dates = prepare_diff_timeframe_data(            train_df, test_daily_df,            train_ratio=TRAIN_RATIO, lookback=LOOKBACK        )                if X_train is None:            print(f"  SKIPPED: Not enough {tf_name} data for {stock}")            for model_type in MODEL_TYPES:                all_results.append({                    'Stock': stock, 'Train_Timeframe': tf_name,                    'Model': model_type, 'MSE': np.nan, 'RMSE': np.nan,                    'MAE': np.nan, 'MAPE (%)': np.nan, 'R2': np.nan,                    'Note': 'Insufficient training data'                })            continue                print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")                for model_type in MODEL_TYPES:            exp_name = f'{EXP_LABEL}_{stock}_{tf_name}'                        y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(                model_type=model_type,                X_train=X_train, y_train=y_train,                X_test=X_test, y_test=y_test,                experiment_name=exp_name,                save_dir=f'models/{EXP_LABEL}',                epochs=EPOCHS, batch_size=BATCH_SIZE            )                        result = {                'Stock': stock, 'Train_Timeframe': tf_name,                'Model': model_type, **metrics            }            all_results.append(result)                        plot_actual_vs_predicted(                test_dates, y_true_inv, y_pred_inv,                model_type, f'{stock}_train_{tf_name}',                EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'            )print("\n\nAll Experiment 3 (80/20) training complete!")

## Results Summary

In [ ]:
# ============================================================# RESULTS TABLE# ============================================================results_df = pd.DataFrame(all_results)print_results_table(results_df, f"Experiment 3 - Different Timeframe (80/20)")results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)print(f"Results saved to results/{EXP_LABEL}_results.csv")

## Visualizations

In [ ]:
# ============================================================# METRICS BAR CHARTS PER STOCK# ============================================================for stock in STOCKS:    stock_df = results_df[results_df['Stock'] == stock].copy()    if stock_df.empty:        continue        for metric in ['RMSE', 'MAE', 'MAPE (%)', 'R2']:        fig, ax = plt.subplots(figsize=(12, 6))                timeframes = stock_df['Train_Timeframe'].unique()        n_tf = len(timeframes)        n_models = len(MODEL_TYPES)        bar_width = 0.8 / n_models        x = np.arange(n_tf)                for i, model_type in enumerate(MODEL_TYPES):            vals = []            for tf in timeframes:                v = stock_df[(stock_df['Model'] == model_type) &                              (stock_df['Train_Timeframe'] == tf)][metric]                vals.append(v.values[0] if len(v) > 0 and not pd.isna(v.values[0]) else 0)                        ax.bar(x + i * bar_width, vals, bar_width,                   label=model_type, color=MODEL_COLORS[model_type],                   edgecolor='white', linewidth=0.5)                ax.set_xlabel('Training Timeframe', fontsize=13)        ax.set_ylabel(metric, fontsize=13)        ax.set_title(f'{EXP_LABEL} | {stock} - {metric} by Timeframe', fontsize=14)        ax.set_xticks(x + bar_width * (n_models - 1) / 2)        ax.set_xticklabels(timeframes, fontsize=11)        ax.legend(fontsize=11)        fig.tight_layout()                fname = f'figures/{EXP_LABEL}/{stock}_{metric}_by_timeframe.png'.replace('(%)', 'pct')        save_fig(fig, fname)print("All bar charts saved!")

In [ ]:
# ============================================================# SUMMARY# ============================================================print("\n" + "="*70)print("  BEST TIMEFRAME PER STOCK (by RMSE)")print("="*70)for stock in STOCKS:    stock_data = results_df[results_df['Stock'] == stock].dropna(subset=['RMSE'])    if stock_data.empty:        continue    best_idx = stock_data['RMSE'].idxmin()    best = stock_data.loc[best_idx]    print(f"  {stock}: {best['Train_Timeframe']} + {best['Model']} "          f"(RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")